#  Phase 2: NoSQL Integration with MongoDB
هذا الملف يحتوي على خطوات إدخال البيانات من ملف إكسل إلى MongoDB باستخدام `pymongo`.

## [1] Install PyMongo

In [ ]:
!pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.6/313.6 kB 21.6 MB/s eta 0:00:00


In [ ]:
!pip install mongomock

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.3 MB/s eta 0:00:00
  Created wheel for sentinels: filename=sentinels-1.0.0-py3-none-any.whl size=3172 sha256=e60a88820daed2a771625b9e94200236e24af7613c7d1f3a60d63c4994f58dc0
  Stored in directory: /root/.cache/pip/wheels/39/e6/05/d0ca91a2c6be3e4b2a6b4e721fe778f9186b9a383ea05300e8
Successfully built sentinels


## [2] Import Libraries

In [ ]:
import pandas as pd
import mongomock

## [3] Load Excel Data

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Updated_Medical_Appointments_Corrected.xlsx to Updated_Medical_Appointments_Corrected.xlsx


In [ ]:
xls_path = "Updated_Medical_Appointments_Corrected.xlsx"

df_patients = pd.read_excel(xls_path, sheet_name="Patients")
df_appointments = pd.read_excel(xls_path, sheet_name="Appointments")
df_conditions = pd.read_excel(xls_path, sheet_name="MedicalConditions")

## [4] Connect to MongoDB

In [ ]:
from mongomock import MongoClient  # هذا هو المهم

client = MongoClient("mongodb://localhost:27017/")
db = client["MedicalDB"]

patients_col = db["Patients"]
appointments_col = db["Appointments"]
conditions_col = db["MedicalConditions"]

## [5] Convert DataFrames to Document Format

In [ ]:
patients_docs = df_patients.to_dict(orient="records")
appointments_docs = df_appointments.to_dict(orient="records")
conditions_docs = df_conditions.to_dict(orient="records")

## [6] Insert Documents into MongoDB

In [ ]:
# حذف البيانات القديمة أولاً
patients_col.delete_many({})
appointments_col.delete_many({})
conditions_col.delete_many({})

# إدخال البيانات الجديدة
patients_col.insert_many(patients_docs)
appointments_col.insert_many(appointments_docs)
conditions_col.insert_many(conditions_docs)

InsertManyResult([ObjectId('6823f89f8d42ac2aed184019'), ObjectId('6823f89f8d42ac2aed18401a'), ObjectId('6823f89f8d42ac2aed18401b'), ObjectId('6823f89f8d42ac2aed18401c'), ObjectId('6823f89f8d42ac2aed18401d'), ObjectId('6823f89f8d42ac2aed18401e'), ObjectId('6823f89f8d42ac2aed18401f'), ObjectId('6823f89f8d42ac2aed184020'), ObjectId('6823f89f8d42ac2aed184021'), ObjectId('6823f89f8d42ac2aed184022'), ObjectId('6823f89f8d42ac2aed184023'), ObjectId('6823f89f8d42ac2aed184024'), ObjectId('6823f89f8d42ac2aed184025'), ObjectId('6823f89f8d42ac2aed184026'), ObjectId('6823f89f8d42ac2aed184027'), ObjectId('6823f89f8d42ac2aed184028'), ObjectId('6823f89f8d42ac2aed184029'), ObjectId('6823f89f8d42ac2aed18402a'), ObjectId('6823f89f8d42ac2aed18402b'), ObjectId('6823f89f8d42ac2aed18402c'), ObjectId('6823f89f8d42ac2aed18402d'), ObjectId('6823f89f8d42ac2aed18402e'), ObjectId('6823f89f8d42ac2aed18402f'), ObjectId('6823f89f8d42ac2aed184030'), ObjectId('6823f89f8d42ac2aed184031'), ObjectId('6823f89f8d42ac2aed1840

## [7] Sample Queries

In [ ]:
# تعديل عمر المريض P1 إلى 70
patients_col.update_one({"PatientId": "P1"}, {"$set": {"Age": 70}})
print("\nUpdated Age for Patient P1:")
print(patients_col.find_one({"PatientId": "P1"}))

# حذف مريض معين (مثلاً P2)
patients_col.delete_one({"PatientId": "P2"})
print("\nRemaining Patients after deleting P2:")
for doc in patients_col.find():
    print(doc)

#  عرض أول 3 مرضى
print(" First 3 Patients:")
for doc in patients_col.find().limit(3):
    print(doc)

#  عرض كل مواعيد المريض P1
print("\n Appointments for Patient P1:")
for doc in appointments_col.find({"PatientId": "P1"}):
    print(doc)

#  عرض المرضى الذين أعمارهم أكبر من 60
print("\n Patients with Age > 38:")
for doc in patients_col.find({"Age": {"$gt": 38}}):
    print(doc)

# عرض الحالات المرضية للمريض P1
print("\n Medical Conditions for Patient P1:")
for doc in conditions_col.find({"PatientId": "P1"}):
    print(doc)


Updated Age for Patient P1:
{'PatientId': 'P1', 'Gender': 'M', 'Age': 70, '_id': ObjectId('6823f89f8d42ac2aed183461')}

Remaining Patients after deleting P2:
{'PatientId': 'P1', 'Gender': 'M', 'Age': 70, '_id': ObjectId('6823f89f8d42ac2aed183461')}
{'PatientId': 'P3', 'Gender': 'M', 'Age': 33, '_id': ObjectId('6823f89f8d42ac2aed183463')}
{'PatientId': 'P4', 'Gender': 'F', 'Age': 34, '_id': ObjectId('6823f89f8d42ac2aed183464')}
{'PatientId': 'P5', 'Gender': 'M', 'Age': 35, '_id': ObjectId('6823f89f8d42ac2aed183465')}
{'PatientId': 'P6', 'Gender': 'F', 'Age': 36, '_id': ObjectId('6823f89f8d42ac2aed183466')}
{'PatientId': 'P7', 'Gender': 'M', 'Age': 37, '_id': ObjectId('6823f89f8d42ac2aed183467')}
{'PatientId': 'P8', 'Gender': 'F', 'Age': 38, '_id': ObjectId('6823f89f8d42ac2aed183468')}
{'PatientId': 'P9', 'Gender': 'M', 'Age': 39, '_id': ObjectId('6823f89f8d42ac2aed183469')}
{'PatientId': 'P10', 'Gender': 'F', 'Age': 30, '_id': ObjectId('6823f89f8d42ac2aed18346a')}
{'PatientId': 'P11', 